# 🩸 Notebook — Modélisation de la COUCHE 2

## Le Sang
Domaine : démographie + épidémiologie + catastrophes humaines.

**Principe central du projet :** prédire la démographie à partir UNIQUEMENT des features environnementales (Couche 1) — montrer que la terre détermine les gens. Les variables socio-éco sont blacklistées pour rester rigoureux.

### Sous-couches modélisées
1. **Démographie** — 7 cibles : natalité, mortalité brut, mortalité infantile, espérance de vie, croissance démo, migration nette, fécondité
2. **Santé / Épidémiologie** — 1 cible : retard de croissance (stunting)
3. **Catastrophes humaines** — 2 cibles : décès et affectés par catastrophes

### Méthodologie
- Données : `data/cleaned/dataset_final_v12_couche1.csv` (8 400 × 741, 240 pays, 8 clusters)
  *(fallback automatique sur v11/v10/v9/v8 si non disponible)*
- Anti-leak strict via `couche2_sang/config.py`
- 2 stratégies comparées par cible :
  - **A.** Modèle GLOBAL (toutes données mondiales)
  - **B.** GLOBAL + feature `cluster` (numéro de cluster climatique en input)
  - *(PerCluster désactivé — jamais utile en V11, gain de temps × 3)*
- Split : `GroupShuffleSplit` par pays (test = pays jamais vus)
- Modèle : XGBoost (500 estimateurs, depth=6, lr=0.05)

### Reproduction
```bash
python couche2_sang/train.py
```
Charge `couche1_planete/models/best_*.joblib` ensuite.

## 1. Imports & configuration partagée

In [ ]:
import os, sys, warnings
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

# Imports projet (config partagée + spécifique couche)
sys.path.insert(0, '..')  # racine du projet depuis sous-dossier
from config_shared import (
    load_dataset, make_preprocessor, make_xgb, select_top_features,
    build_blacklist, split_train_test, TOP_K
)
from config import (
    TARGETS, SUBLAYERS, TARGET_SOURCE, EXTRA_LEAKS,
    YIELD_TARGETS, SOCIO_TARGETS, DISASTER_TARGETS
)

from sklearn.metrics import r2_score, mean_absolute_error
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
os.makedirs('models', exist_ok=True)
os.makedirs('reports', exist_ok=True)

print(f'Cibles 2 : {len(TARGETS)} dans {len(SUBLAYERS)} sous-couches')
for sub, t in SUBLAYERS.items():
    print(f'  {sub:30s} : {len(t)} cibles')

## 2. Chargement du dataset (V12 → V8 fallback)

In [ ]:
# Chargement avec fallback sur la version la plus récente disponible
for path, version in [('../data/cleaned/dataset_final_v12_couche1.csv', 'V12 (suitability + stress + religion + meat)'),
                       ('../data/cleaned/dataset_final_v11_couche1.csv', 'V11 (suitability EcoCrop)'),
                       ('../data/cleaned/dataset_final_v10_couche1.csv', 'V10 (cultures spé + religion + meat)'),
                       ('../data/cleaned/dataset_final_v9_couche1.csv',  'V9 (élevage + énergie + émissions)'),
                       (None, 'V8 default')]:
    if path is None:
        df = load_dataset()
        print(f'✓ Dataset {version} chargé : {df.shape}'); break
    if os.path.exists(path):
        df = load_dataset(path)
        print(f'✓ Dataset {version} chargé : {df.shape}'); break

print(f'   {df["ISO"].nunique()} pays, {df["cluster"].nunique()} clusters')

# Cibles dérivées (si pas déjà créées)
for c in ['disaster_deaths', 'disaster_affected']:
    if c in df.columns and f'target_{c}' not in df.columns:
        df[f'target_{c}'] = np.log1p(df[c].clip(lower=0))
if 'stunting_pct' in df.columns and 'target_stunting' not in df.columns:
    df['target_stunting'] = df['stunting_pct']
if 'Fertility_Rate' in df.columns and 'target_fertility' not in df.columns:
    df['target_fertility'] = df['Fertility_Rate']

# Vérification : toutes les cibles de la couche sont présentes
missing = [t for t in TARGETS if t not in df.columns]
print(f'\nCibles présentes dans df : {len(TARGETS)-len(missing)}/{len(TARGETS)}')
if missing:
    print(f'⚠️  Cibles absentes : {missing}')

## 3. Fonctions d'entraînement (héritées de config_shared)

In [ ]:
def get_bl(target, keep_cluster=False):
    return build_blacklist(
        df, target,
        target_source=TARGET_SOURCE.get(target),
        extra_leaks=EXTRA_LEAKS.get(target, []),
        yield_targets=YIELD_TARGETS,
        socio_targets=SOCIO_TARGETS,
        disaster_targets=DISASTER_TARGETS,
        keep_cluster=keep_cluster,
    )

def train_global(target, keep_cluster=False):
    d = df.dropna(subset=[target]).copy()
    if len(d) < 200: return None
    bl = get_bl(target, keep_cluster=keep_cluster)
    feats = [c for c in d.columns if c not in bl and d[c].dtype != object]
    feats = [c for c in feats if d[c].notna().sum() > 0]
    tr, te = split_train_test(d)
    Xtr_full, Xte_full = tr[feats], te[feats]
    ytr, yte = tr[target], te[target]
    sel = select_top_features(Xtr_full, ytr)
    if keep_cluster and 'cluster' in feats and 'cluster' not in sel:
        sel = ['cluster'] + sel[:-1]
    Xtr, Xte = Xtr_full[sel], Xte_full[sel]
    from sklearn.pipeline import Pipeline
    pipe = Pipeline([('pre', make_preprocessor()), ('model', make_xgb())])
    pipe.fit(Xtr, ytr)
    pred = pipe.predict(Xte)
    return {'r2': r2_score(yte, pred), 'mae': mean_absolute_error(yte, pred),
            'pipe': pipe, 'features': sel, 'n_obs': len(d)}

## 4. Entraînement par sous-couche (Global + Global+Cluster)

In [ ]:
results = []

for sublayer, tgts in SUBLAYERS.items():
    print(f'━━━ {sublayer} ━━━')
    for tgt in tgts:
        if tgt not in df.columns: continue
        res_g  = train_global(tgt, keep_cluster=False)
        if res_g is None: continue
        res_gc = train_global(tgt, keep_cluster=True)
        # Sauvegarde meilleur
        joblib.dump({'pipe': res_gc['pipe'], 'features': res_gc['features']},
                    f'models/best_{tgt}.joblib')
        label = TARGETS[tgt]
        print(f'  🎯 {label:35s} Glob={res_g["r2"]:+.3f}  Glob+C={res_gc["r2"]:+.3f}')
        results.append({
            'Sous-couche': sublayer, 'Cible': label, 'Technique': tgt,
            'R² Global': round(res_g['r2'], 4),
            'R² Global+Cluster': round(res_gc['r2'], 4),
            'MAE': round(res_g['mae'], 3),
            'N obs': res_g['n_obs'],
        })
    print()

out = pd.DataFrame(results)
out.to_csv('reports/results_notebook.csv', index=False)
print('═' * 70)
print(f'📊 RÉSULTATS NOTEBOOK')
print('═' * 70)
print(out.to_string(index=False))

## 5. Visualisation des résultats — performance par cible

In [ ]:
fig, ax = plt.subplots(figsize=(11, max(5, len(out)*0.5)))
out_sorted = out.sort_values('R² Global+Cluster')

# Couleurs par sous-couche
sublayer_colors = {sub: plt.cm.tab10(i) for i, sub in enumerate(SUBLAYERS)}
colors = [sublayer_colors[s] for s in out_sorted['Sous-couche']]

ax.barh(out_sorted['Cible'], out_sorted['R² Global+Cluster'], color=colors, alpha=0.85)
ax.set_xlim(-0.1, 1.0)
ax.axvline(0.5, color='gray', ls='--', alpha=0.5, label='R²=0.5')
ax.axvline(0.7, color='green', ls='--', alpha=0.5, label='R²=0.7')
ax.axvline(0, color='black', lw=0.5)
ax.set_xlabel('R² sur pays jamais vus')

# Légende sous-couches
from matplotlib.patches import Patch
patches = [Patch(color=c, label=s) for s, c in sublayer_colors.items()]
ax.legend(handles=patches + [ax.get_legend_handles_labels()[0][0]],
          labels=list(sublayer_colors.keys()) + ['R²=0.5'], loc='lower right')
ax.set_title(f'Performance par cible — Couche 2', weight='bold', fontsize=14)
plt.tight_layout()
plt.savefig('reports/r2_par_cible.png', dpi=120, bbox_inches='tight')
plt.show()


## 6. Feature importance — top 10 par cible

In [ ]:
TOP_CIBLES = out.sort_values('R² Global+Cluster', ascending=False).head(6)['Technique'].tolist()
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
for ax, tgt in zip(axes.flatten(), TOP_CIBLES):
    path = f'models/best_{tgt}.joblib'
    if not os.path.exists(path): continue
    data = joblib.load(path)
    pipe = data['pipe']
    feats = data['features']
    model = pipe.named_steps['model']
    if not hasattr(model, 'feature_importances_'): continue
    imp = pd.Series(model.feature_importances_, index=feats).sort_values(ascending=True).tail(10)
    colors = ['red' if f=='cluster' else 'steelblue' for f in imp.index]
    ax.barh(imp.index, imp.values, color=colors, alpha=0.85)
    label = TARGETS[tgt]
    ax.set_title(label, weight='bold', fontsize=10)
    ax.tick_params(axis='y', labelsize=8)
plt.suptitle(f'Top 10 features — Couche 2 (rouge = cluster)',
             weight='bold', fontsize=14, y=1.01).replace("2","")
plt.tight_layout()
plt.savefig('reports/feature_importance.png', dpi=120, bbox_inches='tight')
plt.show()


## 7. Diagnostic prédit vs réel — 4 cibles fortes

In [ ]:
DIAG_TARGETS = out.sort_values('R² Global+Cluster', ascending=False).head(4)['Technique'].tolist()
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, tgt in zip(axes, DIAG_TARGETS):
    path = f'models/best_{tgt}.joblib'
    if not os.path.exists(path): continue
    data = joblib.load(path)
    pipe = data['pipe']
    feats = data['features']
    d = df.dropna(subset=[tgt]).copy()
    tr, te = split_train_test(d)
    X_te, y_te = te[feats], te[tgt]
    y_pred = pipe.predict(X_te)
    r2 = r2_score(y_te, y_pred)
    scatter = ax.scatter(y_te, y_pred, c=te['cluster'], cmap='tab10', alpha=0.4, s=10)
    lims = [min(y_te.min(), y_pred.min()), max(y_te.max(), y_pred.max())]
    ax.plot(lims, lims, 'r--', lw=1.5)
    ax.set_xlabel('Réel')
    ax.set_ylabel('Prédit')
    ax.set_title(f'{TARGETS[tgt]}\nR² = {r2:.3f}', weight='bold', fontsize=10)
plt.suptitle(f'Prédit vs Réel — Couche 2 (couleur = cluster)',
             weight='bold', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('reports/diagnostic.png', dpi=120, bbox_inches='tight')
plt.show()


## 8. Synthèse — Couche 2

Tous les modèles sont sauvegardés dans `couche2_*/models/best_*.joblib`.

### Pour charger un modèle :
```python
data = joblib.load('models/best_target_NAME.joblib')
pipe = data['pipe']           # pipeline sklearn complet
features = data['features']   # liste des features utilisées

# Prédire sur de nouvelles données :
prediction = pipe.predict(new_data[features])
```

### Pour la simulation
Ce modèle s'utilise dans le moteur de simulation pour estimer les sorties de cette couche
à partir des features environnementales d'entrée.

> Voir `couche2_*/reports/results.csv` pour le tableau complet.